In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.pydantic_v1 import Field, BaseModel
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
import json
import os

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from typing import List
import base64
import json
import os


# ============================================
# SCHEMAS
# ============================================

class BoundingBox(BaseModel):
    class_id: int = Field(
        description="1 = idle, 0 = not_idle"
    )

    x_center: float = Field(
        description="YOLO normalized x center (0-1)"
    )

    y_center: float = Field(
        description="YOLO normalized y center (0-1)"
    )

    width: float = Field(
        description="YOLO normalized width (0-1)"
    )

    height: float = Field(
        description="YOLO normalized height (0-1)"
    )


class DetectionResponse(BaseModel):
    boxes: List[BoundingBox]


# ============================================
# MODEL
# ============================================

llm = ChatOpenAI(
    model="gpt-4.1",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0
)

structured_llm = llm.with_structured_output(DetectionResponse)


# ============================================
# IMAGE ENCODER
# ============================================

def encode_image(image_path):

    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")


# ============================================
# MAIN
# ============================================

def run_classification(image_path):

    base64_image = encode_image(image_path)

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": """
You are an expert in retail shelf analysis.

Analyze the shelf image and generate YOLO bounding boxes.

Rules:

- Detect ALL shelf facings.
- Each facing must receive ONE bounding box.
- Empty facings:
    class_id = 1 (idle)

- Occupied facings:
    class_id = 0 (not_idle)

Important:

- Bounding boxes must tightly fit the facing area.
- Coordinates MUST be normalized between 0 and 1.
- Return ONLY valid structured output.
- Do not skip visible facings.
- Ignore perspective distortion as much as possible.
"""
            },

            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]
    )

    response = structured_llm.invoke([message])

    return response


# ============================================
# SAVE YOLO TXT
# ============================================

def save_yolo_txt(response, output_path):

    # Se vier dict
    if isinstance(response, dict):
        boxes = response.get("boxes", [])

    # Se vier Pydantic
    else:
        boxes = response.boxes

    with open(output_path, "w") as f:

        for box in boxes:

            # dict
            if isinstance(box, dict):

                line = (
                    f"{box['class_id']} "
                    f"{box['x_center']} "
                    f"{box['y_center']} "
                    f"{box['width']} "
                    f"{box['height']}\n"
                )

            # Pydantic object
            else:

                line = (
                    f"{box.class_id} "
                    f"{box.x_center} "
                    f"{box.y_center} "
                    f"{box.width} "
                    f"{box.height}\n"
                )

            f.write(line)


In [6]:
result = run_classification("C:\\stock_control_deep_learning\\runs\\detect\\predict7\\IMG-20250318-WA0233_jpg.rf.04c94b1e318ec299b6f920300f98701e.jpg")

print(result)

save_yolo_txt(
    result,
    "labels.txt"
)

{}
